In [1]:
# Locate the repo root robustly and make sure the local package is importable.
from pathlib import Path
import sys
import ee

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "geecomposer_v0.1_spec.md").exists() and (candidate / "08_pkg" / "src").exists():
            return candidate
    raise RuntimeError("Could not find repo root from current working directory.")

repo_root = find_repo_root(Path.cwd().resolve())
pkg_src = repo_root / "08_pkg" / "src"

if str(pkg_src) not in sys.path:
    sys.path.insert(0, str(pkg_src))

from geecomposer import initialize, compose, compose_yearly, export_to_drive
from geecomposer.aoi import to_ee_geometry
from geecomposer.transforms.indices import ndvi

aoi_path = repo_root / "01_data" / "case_studies" / "rbmn.geojson"
print("Repo root:", repo_root)
print("AOI path:", aoi_path)
print("Package source:", pkg_src)

Repo root: C:\Users\dev\work\manglaria\repos\geecomposer_dev
AOI path: C:\Users\dev\work\manglaria\repos\geecomposer_dev\01_data\case_studies\rbmn.geojson
Package source: C:\Users\dev\work\manglaria\repos\geecomposer_dev\08_pkg\src


In [2]:
# Uses your already-configured local EE credentials.
# Set authenticate=True only if you want to re-run the interactive auth flow.
initialize(project="manglariars", authenticate=False)

# Simple smoke check that EE is alive.
assert ee.Number(1).getInfo() == 1
print("Earth Engine initialized successfully.")

Earth Engine initialized successfully.


In [3]:
# Normalize the local GeoJSON AOI into an ee.Geometry.
aoi = to_ee_geometry(aoi_path)

# Keep inspection lightweight: bounds are enough for a smoke test.
aoi_bounds = aoi.bounds().getInfo()
aoi_bounds

{'geodesic': False,
 'type': 'Polygon',
 'coordinates': [[[-105.70053541497738, 21.74175903075471],
   [-105.2937762317876, 21.74175903075471],
   [-105.2937762317876, 22.573077683788167],
   [-105.70053541497738, 22.573077683788167],
   [-105.70053541497738, 21.74175903075471]]]}

In [15]:
# Simple optical smoke test: red band median composite with cloud masking.
s2_red = compose(
    dataset="sentinel2",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    mask="s2_cloud_score_plus",
    select="B4",
    reducer="median",
)

s2_red_bands = s2_red.bandNames().getInfo()
s2_red_props = s2_red.getInfo()["properties"]

assert "B4" in s2_red_bands
assert s2_red_props["geecomposer:dataset"] == "sentinel2"
assert s2_red_props["geecomposer:reducer"] == "median"

{"bands": s2_red_bands, "properties": s2_red_props}

{'bands': ['B4'],
 'properties': {'geecomposer:transform': '',
  'geecomposer:collection': 'COPERNICUS/S2_SR_HARMONIZED',
  'geecomposer:reducer': 'median',
  'geecomposer:dataset': 'sentinel2',
  'geecomposer:start': '2025-01-01',
  'geecomposer:end': '2025-12-31'}}

In [16]:
# This validates the built-in transform path in live EE.
s2_ndvi = compose(
    dataset="sentinel2",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    mask="s2_cloud_score_plus",
    transform=ndvi(),
    reducer="max",
)

s2_ndvi_bands = s2_ndvi.bandNames().getInfo()
s2_ndvi_props = s2_ndvi.getInfo()["properties"]

assert "ndvi" in [b.lower() for b in s2_ndvi_bands] or len(s2_ndvi_bands) == 1
assert s2_ndvi_props["geecomposer:transform"] == "ndvi"
assert s2_ndvi_props["geecomposer:dataset"] == "sentinel2"

{"bands": s2_ndvi_bands, "properties": s2_ndvi_props}

{'bands': ['ndvi'],
 'properties': {'geecomposer:transform': 'ndvi',
  'geecomposer:collection': 'COPERNICUS/S2_SR_HARMONIZED',
  'geecomposer:reducer': 'max',
  'geecomposer:dataset': 'sentinel2',
  'geecomposer:start': '2025-01-01',
  'geecomposer:end': '2025-12-31'}}

In [17]:
# Simple radar smoke test: VV median composite with minimal dataset-specific filters.
s1_vv = compose(
    dataset="sentinel1",
    aoi=str(aoi_path),
    start="2025-01-01",
    end="2025-12-31",
    select="VV",
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV"],
        "orbitPass": "ASCENDING",
    },
)

s1_vv_bands = s1_vv.bandNames().getInfo()
s1_vv_props = s1_vv.getInfo()["properties"]

assert "VV" in s1_vv_bands
assert s1_vv_props["geecomposer:dataset"] == "sentinel1"
assert s1_vv_props["geecomposer:reducer"] == "median"

{"bands": s1_vv_bands, "properties": s1_vv_props}

{'bands': ['VV'],
 'properties': {'geecomposer:transform': '',
  'geecomposer:collection': 'COPERNICUS/S1_GRD',
  'geecomposer:reducer': 'median',
  'geecomposer:dataset': 'sentinel1',
  'geecomposer:start': '2025-01-01',
  'geecomposer:end': '2025-12-31'}}

In [19]:
# Sentinel 2 example grouped compose should return one ee.Image per year.
#yearly_ndvi = compose_yearly(
#    years=[2023, 2024, 2025],
#    dataset="sentinel2",
#    aoi=str(aoi_path),
#    mask="s2_cloud_score_plus",
#    transform=ndvi(),
#    reducer="median",
#)
#
#assert set(yearly_ndvi.keys()) == {2023, 2024, 2025}
##print("Year keys:", sorted(yearly_ndvi.keys()))
#
# Inspect one year's metadata to confirm the yearly date derivation.
#year_2025_props = yearly_ndvi[2025].getInfo()["properties"]
#year_2025_props

Year keys: [2023, 2024, 2025]


{'geecomposer:transform': 'ndvi',
 'geecomposer:collection': 'COPERNICUS/S2_SR_HARMONIZED',
 'geecomposer:reducer': 'max',
 'geecomposer:dataset': 'sentinel2',
 'geecomposer:start': '2025-01-01',
 'geecomposer:end': '2026-01-01'}

In [20]:
# Build one Sentinel-1 image per year.
yearly_s1_vv = compose_yearly(
    years=[2023, 2024, 2025],
    dataset="sentinel1",
    aoi=str(aoi_path),
    select="VV",
    reducer="median",
    filters={
        "instrumentMode": "IW",
        "polarizations": ["VV"],
    },
)

assert set(yearly_s1_vv.keys()) == {2023, 2024, 2025}
print("Year keys:", sorted(yearly_s1_vv.keys()))

year_2025_props = yearly_s1_vv[2025].getInfo()["properties"]
year_2025_props

Year keys: [2023, 2024, 2025]


{'geecomposer:transform': '',
 'geecomposer:collection': 'COPERNICUS/S1_GRD',
 'geecomposer:reducer': 'median',
 'geecomposer:dataset': 'sentinel1',
 'geecomposer:start': '2025-01-01',
 'geecomposer:end': '2026-01-01'}

In [ ]:
# Sentinel 2 example. Create the task only. This does NOT start it yet.
# Use a dedicated Drive folder to keep smoke-test exports tidy.
#drive_folder = "geecomposer-dev"
#export_description = "rbmn_s2_ndvi_max_2025_smoke"
#
#task = export_to_drive(
#    image=s2_ndvi,
#    description=export_description,
#    folder=drive_folder,
#    region=str(aoi_path),
#    scale=10,
#)

#task_status = task.status()
#task_status

In [21]:
drive_folder = "geecomposer-dev"
export_description = "rbmn_s1_vv_median_2025_smoke"

s1_2025_task = export_to_drive(
    image=yearly_s1_vv[2025],
    description=export_description,
    folder=drive_folder,
    region=str(aoi_path),
    scale=10,
)

s1_2025_task.status()

{'state': <State.UNSUBMITTED: 'UNSUBMITTED'>}

In [23]:
# Sentinel-1 example. Create the task only. This does NOT start it yet.
# Use a dedicated Drive folder to keep smoke-test exports tidy.

drive_folder = "geecomposer-dev"
export_description = "rbmn_s1_vv_median_2025_smoke"

s1_task = export_to_drive(
    image=yearly_s1_vv[2025],
    description=export_description,
    folder=drive_folder,
    region=str(aoi_path),
    scale=10,
)

s1_task_status = s1_task.status()
s1_task_status

{'state': <State.UNSUBMITTED: 'UNSUBMITTED'>}

In [25]:
DO_START_EXPORT = True

if DO_START_EXPORT:
    s1_task.start()
    print("Sentinel-1 export task started.")
else:
    print("Task not started. Set DO_START_EXPORT = True to launch the export.")

s1_task.status()

Sentinel-1 export task started.


{'state': 'READY',
 'description': 'rbmn_s1_vv_median_2025_smoke',
 'priority': 100,
 'creation_timestamp_ms': 1776181077913,
 'update_timestamp_ms': 1776181077913,
 'start_timestamp_ms': 0,
 'task_type': 'EXPORT_IMAGE',
 'id': 'FTWKX4SMB5VRFFP6BKN2QKZE',
 'name': 'projects/manglariars/operations/FTWKX4SMB5VRFFP6BKN2QKZE'}

In [26]:
# Run this only after starting the task. It will poll until completion or timeout.
import time
from pprint import pprint

MAX_POLLS = 100
SLEEP_SECONDS = 20

for i in range(MAX_POLLS):
    status = s1_task.status()
    state = status.get("state", "UNKNOWN")
    print(f"Poll {i+1}/{MAX_POLLS}: {state}")
    
    if state in {"COMPLETED", "FAILED", "CANCELLED"}:
        pprint(status)
        break
    
    time.sleep(SLEEP_SECONDS)
else:
    print("Polling limit reached; task may still be running.")
    pprint(s1_task.status())

Poll 1/100: READY
Poll 2/100: READY
Poll 3/100: READY
Poll 4/100: READY
Poll 5/100: READY
Poll 6/100: READY
Poll 7/100: READY
Poll 8/100: READY
Poll 9/100: READY
Poll 10/100: READY
Poll 11/100: READY
Poll 12/100: READY
Poll 13/100: READY
Poll 14/100: RUNNING
Poll 15/100: RUNNING
Poll 16/100: RUNNING
Poll 17/100: RUNNING
Poll 18/100: RUNNING
Poll 19/100: RUNNING
Poll 20/100: RUNNING
Poll 21/100: RUNNING
Poll 22/100: RUNNING
Poll 23/100: RUNNING
Poll 24/100: RUNNING
Poll 25/100: RUNNING
Poll 26/100: RUNNING
Poll 27/100: RUNNING
Poll 28/100: RUNNING
Poll 29/100: RUNNING
Poll 30/100: RUNNING
Poll 31/100: RUNNING
Poll 32/100: RUNNING
Poll 33/100: RUNNING
Poll 34/100: RUNNING
Poll 35/100: RUNNING
Poll 36/100: RUNNING
Poll 37/100: RUNNING
Poll 38/100: RUNNING
Poll 39/100: RUNNING
Poll 40/100: RUNNING
Poll 41/100: RUNNING
Poll 42/100: RUNNING
Poll 43/100: RUNNING
Poll 44/100: RUNNING
Poll 45/100: RUNNING
Poll 46/100: RUNNING
Poll 47/100: RUNNING
Poll 48/100: RUNNING
Poll 49/100: RUNNING
Poll 5

In [13]:
# Lightweight summary of what worked in the notebook.
summary = {
    "aoi_loaded": True,
    "sentinel2_red_bands": s2_red_bands,
    "sentinel2_ndvi_bands": s2_ndvi_bands,
    "sentinel1_vv_bands": s1_vv_bands,
    "yearly_keys": sorted(yearly_ndvi.keys()),
    "export_description": export_description,
    "task_state": task.status().get("state"),
}
summary

{'aoi_loaded': True,
 'sentinel2_red_bands': ['B4'],
 'sentinel2_ndvi_bands': ['ndvi'],
 'sentinel1_vv_bands': ['VV'],
 'yearly_keys': [2022, 2023, 2024],
 'export_description': 'rbmn_s2_ndvi_max_2024_smoke',
 'task_state': 'COMPLETED'}

In [14]:
# Sentinel 1 raster failed, check:
from pprint import pprint
pprint(s1_task.status())

NameError: name 's1_task' is not defined